# Aspect-based evaluation with ModelRadar

Aggregate error (mean SMAPE over all series and horizons) hides *where* a model fails. **ModelRadar** slices a Nixtla-style cross-validation frame by:

- overall and per-series scores
- forecast horizon (cumulative, and first vs. last step)
- hard unique IDs and expected shortfall (CVaR)
- ROPE win / draw / loss vs. a reference model
- anomalies and arbitrary groups

Cerqueira, V., Roque, L., & Soares, C. (2025). Modelradar: aspect-based forecast evaluation. *Machine Learning*, 114(10), 229.

In [ ]:
import numpy as np
import pandas as pd
from utilsforecast.losses import mae, smape

from metaforecast.evaluation.aspects import ModelRadar

Build a small synthetic CV frame (same columns you get from `NeuralForecast.cross_validation` / `StatsForecast.cross_validation`).

In [ ]:
rng = np.random.default_rng(0)
rows = []
for i, uid in enumerate(["A", "B", "C", "D", "E", "F", "G", "H"]):
    n = 12
    t = np.arange(n)
    y = 10 + 0.15 * t + 2 * np.sin(2 * np.pi * t / 12) + rng.normal(0, 0.4, n)
    naive = np.concatenate([[y[0]], y[:-1]])
    hits = y + rng.normal(0, 0.25, n)
    mlp = y + rng.normal(0, 1.2, n)
    rows.append(
        pd.DataFrame(
            {
                "unique_id": uid,
                "ds": pd.date_range("2020-01-01", periods=n, freq="ME"),
                "cutoff": pd.Timestamp("2019-12-01"),
                "y": y,
                "SeasonalNaive": naive,
                "NHITS": hits,
                "MLP": mlp,
                "is_anomaly": (np.abs(rng.normal(0, 1, n)) > 1.6).astype(int),
                "seas_str": "Seasonal" if i % 2 == 0 else "Non-seasonal",
            }
        )
    )

cv = pd.concat(rows, ignore_index=True)
cv.head()

Fit ModelRadar. `hardness_reference` defines hard series; `ratios_reference` is the ROPE champion.

In [ ]:
radar = ModelRadar(
    cv_df=cv,
    metrics=[smape, mae],
    model_names=["NHITS", "MLP", "SeasonalNaive"],
    hardness_reference="SeasonalNaive",
    ratios_reference="NHITS",
    rope=10,
)
radar.model_order

In [ ]:
overall = radar.evaluate()
overall

In [ ]:
err = radar.evaluate(keep_uids=True)
err.head()

Hard unique IDs, accuracy on those series, and expected shortfall (average error in the worst tail).

In [ ]:
hard = radar.uid_accuracy.get_hard_uids(err)
print("hard uids:", radar.uid_accuracy.hard_uid)
pd.concat(
    [
        overall,
        radar.uid_accuracy.accuracy_on_hard(err),
        radar.uid_accuracy.expected_shortfall(err),
    ],
    axis=1,
)

ROPE: share of series where each model beats / draws / loses to NHITS by more than 10%.

In [ ]:
radar.rope.get_winning_ratios(err)

Horizon bounds, anomalies, and a grouping column.

In [ ]:
radar.evaluate_by_horizon_bounds()

In [ ]:
radar.evaluate_by_anomaly(mode="observations")

In [ ]:
radar.evaluate_by_group("seas_str")

The per-UID score matrix `err` is the same shape `ActiveTesting.select` expects:

```python
from metaforecast.coseal import ActiveTesting

ActiveTesting(max_trials=2).select(err)
```